In [1]:
import adios4dolfinx
import numpy as np
import matplotlib.pyplot as plt
import pyvista as pv

from dolfinx.fem.petsc import LinearProblem
from ufl import sqrt, inner, TestFunction, TrialFunction, dot, grad, CellDiameter, dx
from pathlib import Path
from mpi4py import MPI
from dolfinx import mesh, fem, io, plot
from basix.ufl import element


In [2]:
wind_file = Path("../inverse_AD/wind_data/velocity.bp")
domain = adios4dolfinx.read_mesh(wind_file, MPI.COMM_WORLD)
facet_tags = adios4dolfinx.read_meshtags(wind_file, domain, meshtag_name="facet_tags")


topology, cell_type, geom = plot.vtk_mesh(domain) 
grid = pv.UnstructuredGrid(topology, cell_type, geom)

dom_cell = domain.basix_cell()
V_wind = fem.functionspace(domain, element("Lagrange", dom_cell, 1, shape=(domain.topology.dim,)))

u_true = fem.Function(V_wind)
adios4dolfinx.read_function(wind_file, u_true, name="velocity")